In [ ]:
#Conexão com o Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import json

def get_news_content(links_file):
    # Carrega o arquivo com os links e retorna uma lista
    with open(links_file, 'r') as file:
        links = set(file.readlines())
    news_contents = []

    for link in links:
        link = link.strip()  # Limpa a URL, se necessário
        response = requests.get(link)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # Localiza o elemento que contém o conteúdo do artigo
            article_content = soup.find('div', class_='article__content')
            if article_content:
                paragraphs = article_content.find_all('p', class_='paragraph-elevate inline-placeholder vossi-paragraph_elevate')
                # Concatena o texto de todos os parágrafos encontrados
                content = ' '.join(p.get_text(strip=True) for p in paragraphs)
                news_contents.append(content)
            else:
                news_contents.append("Conteúdo não encontrado.")
        else:
            news_contents.append(f"Falha ao extrair o conteúdo da notícia {link}: {response.status_code}")

    # Salva o conteúdo em um arquivo JSON
    with open('news_contents.json', 'w', encoding="utf-8") as json_file:
        json.dump({"news_content": news_contents}, json_file, ensure_ascii=False)

# Chamada da função para extrair os conteúdos
get_news_content('/content/drive/MyDrive/CNN_links')


In [8]:
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

In [9]:
from googleapiclient.http import MediaFileUpload

file_metadata = {
  'name': 'CNN_news_contents',
  'mimeType': 'text/plain'
}
media = MediaFileUpload('/content/news_contents.json',
                        mimetype='text/plain',
                        resumable=True)
created = drive_service.files().create(body=file_metadata,
                                       media_body=media,
                                       fields='id').execute()
print('File ID: {}'.format(created.get('id')))

File ID: 12K-HshSseF69SZeqfBkO96J4CPayCUIY
